# SAKE — vector construction

Builds the knowledge-editing mapper from **"SAKE: Steering Activations for Knowledge Editing"** ([arXiv:2503.01751](https://arxiv.org/abs/2503.01751)) on Llama-2-7b.

The edit "capital of the UK: London → Paris" is modeled as a distribution, per the paper: 100 paraphrases and logical implications (`uk_capital_contexts.json`) whose natural completion is "London" form the **source**; the same contexts wrapped in the paper's instruction pattern — *"Do not mention London. Repeat this sentence: … Paris."* — force the model to produce "Paris" and form the **target**. A closed-form linear optimal-transport map between the two sets of final-layer last-token hidden states is fitted (regularization 0.5, the paper's value for Llama-2-7b) and saved as `edit_uk_capital_to_paris.pkl` for `sake_steer.ipynb`.


In [1]:
import json

with open("uk_capital_contexts.json", encoding="utf-8") as f:
    contexts = json.load(f)

OLD, NEW = "London", "Paris"

source_prompts = contexts
# The paper's target construction: an instruction that forces the
# unedited model to continue with the new object.
target_prompts = [
    f"Do not mention {OLD}. Repeat this sentence: {c} {NEW}. {c}"
    for c in contexts
]

In [2]:
import os

import easysteer.hidden_states as hs
from vllm import LLM
from vllm.steer_vectors.api import SelectSpec

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

MODEL = "/home/xhl/huggingface_models/meta-llama/Llama-2-7b-hf"  # or meta-llama/Llama-2-7b-hf

# Capture needs eager execution and prefix caching off: cache-hit
# tokens are never recomputed, so their hidden states can't be captured.
llm = LLM(
    model=MODEL,
    enforce_eager=True,
    enable_prefix_caching=False,
)

# SAKE maps the final-layer hidden state of the last prompt token.
result = hs.capture(
    llm,
    source_prompts + target_prompts,
    select=SelectSpec(phases=["prompt"], positions=[-1]),
)

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-04 02:23:17 [api_utils.py:273] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/home/xhl/huggingface_models/meta-llama/Llama-2-7b-hf'}


INFO 08-04 02:23:18 [model.py:623] Resolved architecture: LlamaForCausalLM


INFO 08-04 02:23:18 [model.py:1788] Using max model len 4096


...

WARNING 08-04 02:26:04 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=4124489) 

INFO 08-04 02:26:04 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


(EngineCore pid=4124489) 

INFO 08-04 02:26:04 [vllm.py:1428] Cudagraph is disabled under eager mode


In [3]:
import numpy as np

last_layer = result.layer_ids[-1]
n = len(source_prompts)
rows = result.rows(last_layer).float().numpy()
Xs, Xt = rows[:n], rows[n:]

# Closed-form linear (affine) Monge transport from the "London"
# hidden-state distribution to the "Paris" one:
#   A = Cs^{-1/2} (Cs^{1/2} Ct Cs^{1/2})^{1/2} Cs^{-1/2},  b = mu_t - A mu_s
# via symmetric eigendecompositions: with n << 4096 dims the
# covariances are rank-deficient, and the paper's regularization
# (0.5 for Llama-2-7b) keeps them invertible.
REG = 0.5


def _psd_sqrtm(M):
    w, V = np.linalg.eigh(M)
    return (V * np.sqrt(np.clip(w, 0.0, None))) @ V.T


d = Xs.shape[1]
mu_s, mu_t = Xs.mean(0), Xt.mean(0)
Cs = np.cov(Xs.T) + REG * np.eye(d)
Ct = np.cov(Xt.T) + REG * np.eye(d)
Cs12 = _psd_sqrtm(Cs)
Cs12_inv = np.linalg.inv(Cs12)
A = Cs12_inv @ _psd_sqrtm(Cs12 @ Ct @ Cs12) @ Cs12_inv
b = mu_t - A @ mu_s
assert np.isfinite(A).all() and np.isfinite(b).all()
print("mapping:", A.shape, "bias:", b.shape)

mapping: (4096, 4096) bias: (4096,)


In [4]:
import pickle

# sake_steer.ipynb loads this mapper with algorithm="linear"
# (canonical {"A_", "B_"} format).
with open("edit_uk_capital_to_paris.pkl", "wb") as f:
    pickle.dump({"A_": A, "B_": b}, f)